> `oeai_mod_bromcom_bronze_odata` - Bronze notebook for Bromcom + OData connection

In [ ]:
%run oeai_mod_bromcom_env_var

In [ ]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_bromcom_bronze_odata.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

In [ ]:
# HACF guard — raise hard if connection is not "odata"
if (connection or "").strip().lower() != "odata":
    oeai.log.critical(f"HACF: connection='{connection}' != 'odata'. Intentionally stopping execution.")
    raise RuntimeError(
        f"HACF: connection='{connection}' != 'odata'. Intentionally stopping execution."
    )
oeai.log.debug("Guard passed: connection == 'odata'. Proceeding…")

In [ ]:
import requests
import xml.etree.ElementTree as ET
from requests.auth import HTTPBasicAuth

# --- OData configuration ---
metadata_url = "https://vision.bromcom.com/OData/$metadata"
username = bromcom_odata_feed_username
password = bromcom_odata_feed_password

# --- Fetch metadata ---
response = requests.get(metadata_url, auth=HTTPBasicAuth(username, password))
response.raise_for_status()  # raise error if 4xx or 5xx
metadata_xml = response.text

# --- Parse XML and detect namespaces dynamically ---
root = ET.fromstring(metadata_xml)

# Function to extract namespace from XML tag
def get_namespace(tag):
    if tag[0] == '{':
        return tag[1:].split('}')[0]
    return ''

# Detect all namespaces in use
namespaces = set()
for elem in root.iter():
    ns = get_namespace(elem.tag)
    if ns:
        namespaces.add(ns)

print("Detected namespaces:", namespaces)

# We'll search all possible elements named 'EntitySet' (regardless of prefix)
entity_sets = []
for elem in root.iter():
    if elem.tag.endswith('EntitySet'):
        entity_sets.append({
            "TableName": elem.attrib.get("Name"),
            "EntityType": elem.attrib.get("EntityType")
        })

# --- Convert to Spark DataFrame ---
spark_df = spark.createDataFrame(entity_sets)

print(f"✅ Found {spark_df.count()} tables in the OData feed")
display(spark_df)


In [ ]:
oeai.log.end_block()
oeai.log.checkpoint("init")

In [ ]:
oeai.log.start_block("Processing")

In [ ]:
# OData → Delta (Bronze) loader
# Full + Filtered entities (weekly range for BehaviourEvents; weekly-upper-bound for SessionAttendance)
#
# What this does
# - Lets you specify some entities to pull EVERYTHING, and others to LIMIT by a date column
# - For filtered entities:
#     * BehaviourEvents → WEEKLY [start, end) windows with OData $filter ge/lt (both bounds)
#     * SessionAttendance → WEEKLY steps using only an exclusive upper bound (… lt <cutoff>)
# - Paginates robustly (nextLink and $skip)
# - Prints clear progress while running
#
# Prereqs:
#   - spark session available
#   - bromcom_odata_feed_username, bromcom_odata_feed_password defined
#   - bronze_path defined (e.g., "/mnt/bronze")

import requests, json, time
from requests.auth import HTTPBasicAuth
from urllib.parse import urlencode, urljoin
from pyspark.sql import functions as F
from datetime import datetime, timedelta

# -----------------------
# Global Config
# -----------------------
BASE_URL      = "https://vision.bromcom.com/OData"
PAGE_SIZE     = 10000     # HTTP page size; will auto-backoff if needed
DELAY_SEC     = 0.0       # 0.2–0.5 if you hit rate limits
WEEK_STRIDE_DAYS = 7      # change if Bromcom batches differently

# Optional (usually not needed if we partition properly):
# spark.conf.set("spark.rpc.message.maxSize", "512")  # MB

username = bromcom_odata_feed_username
password = bromcom_odata_feed_password


# Derived list of all entities to process in this run
ALL_ENTITIES = FULL_ENTITIES + list(FILTERED_ENTITIES.keys())

# -----------------------
# HTTP session
# -----------------------
session = requests.Session()
session.auth = HTTPBasicAuth(username, password)
session.headers.update({"Accept": "application/json"})

# -----------------------
# Helpers
# -----------------------
def _extract_rows(payload: dict):
    # v4: {"value":[...]}; v2: {"d":{"results":[...]}}
    if isinstance(payload, dict):
        if isinstance(payload.get("value"), list):
            return payload["value"]
        d = payload.get("d")
        if isinstance(d, dict) and isinstance(d.get("results"), list):
            return d["results"]
    return []

def _next_link(payload: dict):
    # v4/v3/v2 variants
    if not isinstance(payload, dict):
        return None
    return (
        payload.get("@odata.nextLink")
        or payload.get("odata.nextLink")
        or (payload.get("d", {}) if isinstance(payload.get("d"), dict) else {}).get("__next")
        or payload.get("__next")
        or payload.get("nextLink")
    )

def _sanitize_record(rec: dict):
    # Drop metadata keys and make columns Spark-friendly
    clean = {}
    for k, v in rec.items():
        if isinstance(k, str) and k.startswith("@"):
            continue
        clean[k.replace(".", "_")] = v
    return clean

def _get_page(url: str):
    resp = session.get(url)
    resp.raise_for_status()
    return resp.json()

def _build_filter_range_variants(date_col: str, start_dt: datetime, end_dt: datetime):
    """
    Build OData range filter variants with inclusive lower bound and exclusive upper bound:
      start <= col < end
    Variants tried:
      1) v2 datetime:         col ge datetime'YYYY-MM-DDTHH:MM:SS' and col lt datetime'...'
      2) v4 instant (Z):      col ge YYYY-MM-DDTHH:MM:SSZ         and col lt YYYY-MM-DDTHH:MM:SSZ
      3) v2 datetimeoffset:   col ge datetimeoffset'...Z'         and col lt datetimeoffset'...Z'
    """
    s_no_tz = start_dt.strftime("%Y-%m-%dT%H:%M:%S")
    e_no_tz = end_dt.strftime("%Y-%m-%dT%H:%M:%S")
    s_z     = s_no_tz + "Z"
    e_z     = e_no_tz + "Z"
    return [
        f"({date_col} ge datetime'{s_no_tz}' and {date_col} lt datetime'{e_no_tz}')",
        f"({date_col} ge {s_z} and {date_col} lt {e_z})",
        f"({date_col} ge datetimeoffset'{s_z}' and {date_col} lt datetimeoffset'{e_z}')",
    ]

def _build_before_variants(date_col: str, end_dt: datetime):
    """
    Build $filter variants with an exclusive upper bound only:
       {date_col} lt <end>
    We try OData v2 and v4 shapes.
    """
    e_no_tz = end_dt.strftime("%Y-%m-%dT%H:%M:%S")
    e_z     = e_no_tz + "Z"
    return [
        f"({date_col} lt datetime'{e_no_tz}')",
        f"({date_col} lt {e_z})",
        f"({date_col} lt datetimeoffset'{e_z}')",
    ]

def iter_daily_windows(start_inclusive: datetime, end_exclusive: datetime):
    """
    Yield [start, end) UTC daily windows, walking backwards from end_exclusive to start_inclusive.
    """
    end = end_exclusive
    while end > start_inclusive:
        start = max(start_inclusive, end - timedelta(days=1))
        yield (start, end)
        end = start

def iter_weekly_windows(start_inclusive: datetime, end_exclusive: datetime):
    """
    Yield [start, end) UTC weekly windows walking backwards.
    """
    end = end_exclusive
    while end > start_inclusive:
        start = max(start_inclusive, end - timedelta(days=WEEK_STRIDE_DAYS))
        yield (start, end)
        end = start

def fetch_all(entity: str, page_size: int = PAGE_SIZE, filter_exprs=None):
    """
    1) First, try server-driven paging (nextLink). If present, follow it.
       - If filter_exprs is provided (list of possible $filter strings), try them in order until one works.
    2) If no nextLink appears, fall back to deterministic $skip/$top.
       - On 400, progressively reduce $top; final fallback: drop $top (server default page size).
    Yields sanitized records.
    """
    base = f"{BASE_URL.rstrip('/')}/{entity}"

    # ---- First request (with or without filter) ----
    tried = []
    first_payload = None
    first_rows = []
    first_url_used = None
    accepted_filter = None  # capture the exact working filter for reuse

    filter_exprs = filter_exprs or [None]  # ensure iterable

    for fexpr in filter_exprs:
        params = {"$top": page_size}
        if fexpr:
            params["$filter"] = fexpr
        url = f"{base}?{urlencode(params)}"
        try:
            # print(f"  → Fetching first page for {entity} (top={page_size})" + (f", filter={fexpr}" if fexpr else ""))
            oeai.log.debug(
                f"  → Fetching first page for {entity} (top={page_size})" + (f", filter={fexpr}" if fexpr else ""),
                entity=entity,
                page_size=page_size,
                fexpr=fexpr,
            )
            payload = _get_page(url)
            rows = _extract_rows(payload)
            first_payload, first_rows, first_url_used = payload, rows, url
            accepted_filter = fexpr  # remember which one actually worked
            break
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else "?"
            tried.append((fexpr, code))
            # print(f"    ⚠️  First request failed for {entity} (status {code}) with filter={fexpr}; trying next variant...")
            oeai.log.warning(
                f"First request failed for {entity} (status {code}) with filter={fexpr}; trying next variant...",
                entity=entity,
                page_size=page_size,
                fexpr=fexpr,
            )
            continue

    if first_payload is None:
        oeai.log.error(
                f"Unable to fetch first page for {entity} with any $filter variant tried: {tried}",
                entity=entity,
                tried=tried,
            )
        raise RuntimeError(f"Unable to fetch first page for {entity} with any $filter variant tried: {tried}")

    # Yield first page
    for r in first_rows:
        yield _sanitize_record(r)

    # ---- Follow nextLink if present (often uses $skiptoken internally) ----
    nxt = _next_link(first_payload)
    if nxt:
        url = urljoin(first_url_used, nxt)
        page_count = 1
        while url:
            page_count += 1
            print(f"    ↳ nextLink page {page_count} for {entity}")
            payload = _get_page(url)
            page = _extract_rows(payload)
            if not page:
                break
            for r in page:
                yield _sanitize_record(r)
            next_url = _next_link(payload)
            url = urljoin(url, next_url) if next_url else None
            if DELAY_SEC:
                time.sleep(DELAY_SEC)
        return

    # ---- No nextLink -> deterministic $skip loop, reusing the EXACT accepted filter variant ----
    skip = len(first_rows)
    current_top = page_size
    page_count = 1

    while True:
        q = {"$skip": skip}
        if current_top is not None:
            q["$top"] = current_top
        if accepted_filter:
            q["$filter"] = accepted_filter
        url = f"{base}?{urlencode(q)}"
        # print(f"    ↳ skip page {page_count} for {entity} (skip={skip}, top={current_top if current_top is not None else 'default'})")
        oeai.log.debug(
            f"skip page {page_count} for {entity} (skip={skip}, top={current_top if current_top is not None else 'default'})",
            page_count=page_count,
            entity=entity,
            skip=skip,
            current_top=current_top,
            )

        try:
            payload = _get_page(url)
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                if current_top is not None and current_top > 100:
                    current_top = max(100, current_top // 2)
                    # print(f"      ⚠️  400 Bad Request – reducing page size to {current_top} and retrying")
                    oeai.log.warning(
                        f"400 Bad Request – reducing page size to {current_top} and retrying",
                        entity=entity,
                        current_top=current_top,
                        )
                    continue  # retry loop with smaller top
                if current_top is not None:
                    # print(f"      ⚠️  final fallback – trying without $top for {entity}")
                    oeai.log.warning(
                        f"final fallback – trying without $top for {entity}",
                        entity=entity,
                        current_top=current_top,
                        )
                    current_top = None
                    continue  # retry loop without top
            raise

        page = _extract_rows(payload)
        if not page:
            break

        for r in page:
            yield _sanitize_record(r)

        # If server introduces a nextLink mid-run, switch to following it
        nxt = _next_link(payload)
        if nxt:
            url = urljoin(url, nxt)
            while url:
                page_count += 1
                print(f"    ↳ switching to nextLink (page {page_count}) for {entity}")
                oeai.log.debug(
                    f"switching to nextLink (page {page_count}) for {entity}",
                    page_count=page_count,
                    entity=entity,
                    )
                payload = _get_page(url)
                page = _extract_rows(payload)
                if not page:
                    break
                for r in page:
                    yield _sanitize_record(r)
                next_url = _next_link(payload)
                url = urljoin(url, next_url) if next_url else None
                if DELAY_SEC:
                    time.sleep(DELAY_SEC)
            break

        skip += len(page)
        page_count += 1
        if current_top is not None and len(page) < current_top:
            break
        if DELAY_SEC:
            time.sleep(DELAY_SEC)

# ---------- NEW: safe JSON→DF helper to avoid huge task payloads ----------
def _json_lines_to_df(json_lines, target_rows_per_partition: int = 50_000, min_partitions: int = 128):
    """
    Create a Spark DF from an in-memory list of JSON strings without blowing up task sizes.
    - Splits into many partitions so each partition carries ~target_rows_per_partition rows.
    - Forces a minimum number of partitions to avoid giant RPC messages.
    """
    n = len(json_lines)
    if n == 0:
        # Shouldn't be hit because callers guard on empties, but return a valid empty DF if it happens.
        return spark.createDataFrame([], schema="dummy STRING").limit(0)

    num_slices = max(min_partitions, (n + target_rows_per_partition - 1) // target_rows_per_partition)
    rdd = spark.sparkContext.parallelize(json_lines, numSlices=num_slices)
    return (
        spark.read
             .option("samplingRatio", "1.0")
             .option("multiLine", "false")
             .option("columnNameOfCorruptRecord", "_corrupt_record")
             .json(rdd)
    )

# -----------------------
# Main: load -> Spark JSON -> Delta (single write per entity)
# -----------------------
# print(f"📅 Bounded fetch start (UTC): {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')}")
oeai.log.debug(f"Bounded fetch start (UTC): {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')}")
results = []

for entity in ALL_ENTITIES:
    mapped_name = entity_mappings.get(entity, entity)
    out_path = f"{bronze_path.rstrip('/')}/{mapped_name}"

    # print(f"\n🚀 Starting entity: {entity} → {mapped_name}")
    oeai.log.info(
        f"Starting entity: {entity} → {mapped_name}",
        entity=entity,
        mapped_name=mapped_name,
        )

    if entity in FILTERED_ENTITIES:
        cfg = FILTERED_ENTITIES[entity]
        date_col = cfg["date_col"]
        mode     = cfg.get("mode", "weekly_range")

        # End-exclusive = start of (UTC) tomorrow so we include "today"
        utc_today_start = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0)
        end_exclusive = utc_today_start + timedelta(days=1)

        if mode == "weekly_range":
            # --- BehaviourEvents path: weekly, both bounds ge/lt ---
            # print(f"   Applying bounded WEEKLY RANGE on {date_col} "
            #       f"[start, end) with ge/lt from {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')} "
            #       f"to {end_exclusive.strftime('%Y-%m-%d %H:%M:%S')}")
            oeai.log.debug(
                f"   Applying bounded WEEKLY RANGE on {date_col} "
                f"[start, end) with ge/lt from {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')} "
                f"to {end_exclusive.strftime('%Y-%m-%d %H:%M:%S')}",
                entity=entity,
                mapped_name=mapped_name,
                mode=mode,
                date_col=date_col,
            )

            all_rows = []
            wk_idx = 0
            for start_dt, end_dt in iter_weekly_windows(BOUND_START_DATE, end_exclusive):
                wk_idx += 1
                range_variants = _build_filter_range_variants(date_col, start_dt, end_dt)
                # print(f"     📦 Week {wk_idx}: [{start_dt.strftime('%Y-%m-%d %H:%M:%S')} → {end_dt.strftime('%Y-%m-%d %H:%M:%S')})")
                wk_rows = list(fetch_all(entity, page_size=PAGE_SIZE, filter_exprs=range_variants))
                # print(f"        • Retrieved {len(wk_rows)} rows")
                oeai.log.debug(
                    f"Week {wk_idx}: [{start_dt.strftime('%Y-%m-%d %H:%M:%S')} → {end_dt.strftime('%Y-%m-%d %H:%M:%S')}]"
                    f" Retrieved {len(wk_rows)} rows",
                    entity=entity,
                    mapped_name=mapped_name,
                    mode=mode,
                    date_col=date_col,
                    wk_idx=wk_idx,
                    start_dt=start_dt,
                    end_dt=end_dt,
                    wk_rows=wk_rows,
                )

                if wk_rows:
                    all_rows.extend(wk_rows)

            if not all_rows:
                # print(f"⚠️  No data returned for {entity} in the bounded period")
                oeai.log.debug(
                    f"⚠️  No data returned for {entity} in the bounded period",
                    entity=entity,
                    mapped_name=mapped_name,
                    mode=mode,
                )

                results.append({"table": entity, "saved_as": mapped_name, "rows_written": 0, "path": out_path, "status": "empty"})
                continue

            # One write at the end for the entire bounded period
            json_lines = [json.dumps(rec, default=str) for rec in all_rows]
            df = _json_lines_to_df(json_lines)

            if "_corrupt_record" in df.columns:
                df = df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

            # Guard (keeps exactly within global bounds; harmless if already precise)
            df = df.filter(
                (F.col(date_col) >= F.lit(BOUND_START_DATE.strftime("%Y-%m-%dT%H:%M:%S"))) &
                (F.col(date_col) <  F.lit(end_exclusive.strftime("%Y-%m-%dT%H:%M:%S")))
            )

            df = df.dropDuplicates().withColumn("_ingested_at", F.current_timestamp())

            # print(f"💾 Writing (bounded weekly-range) {entity} → {mapped_name} (mode=overwrite)")
            oeai.log.debug(
                f"Writing (bounded weekly-range) {entity} → {mapped_name} (mode=overwrite)",
                entity=entity,
                mapped_name=mapped_name,
            )
            (
                df.write
                  .format("delta")
                  .mode("overwrite")
                  .option("overwriteSchema", "true")
                  .save(out_path)
            )
            rows_written = spark.read.format("delta").load(out_path).count()
            # print(f"✅ Completed {entity} → {mapped_name} (rows written: {rows_written:,})")
            oeai.log.info(
                f"Completed {entity} → {mapped_name} (rows written: {rows_written:,})",
                entity=entity,
                mapped_name=mapped_name,
                rows_written=rows_written,
            )
            results.append({
                "table": entity,
                "saved_as": mapped_name,
                "rows_written": rows_written,
                "path": out_path,
                "status": "ok" if rows_written > 0 else "empty"
            })

        elif mode == "weekly_upper_bound":
            # --- SessionAttendance path: weekly, upper-bound-only ---
            # print(f"   Applying bounded WEEKLY 'before' windows on {date_col} using "
            #       f"{date_col} lt <cutoff>, stepping back {WEEK_STRIDE_DAYS} days at a time.\n"
            #       f"   From {end_exclusive.strftime('%Y-%m-%d %H:%M:%S')} (exclusive) "
            #       f"back to {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')} (inclusive)")
            oeai.log.debug(
                f"   Applying bounded WEEKLY 'before' windows on {date_col} using "
                f"{date_col} lt <cutoff>, stepping back {WEEK_STRIDE_DAYS} days at a time.\n"
                f"   From {end_exclusive.strftime('%Y-%m-%d %H:%M:%S')} (exclusive) "
                f"back to {BOUND_START_DATE.strftime('%Y-%m-%d %H:%M:%S')} (inclusive)",
                entity=entity,
                mapped_name=mapped_name,
                mode=mode,
                date_col=date_col,
            )

            all_rows = []
            week_idx = 0
            for start_dt, end_dt in iter_weekly_windows(BOUND_START_DATE, end_exclusive):
                week_idx += 1
                before_variants = _build_before_variants(date_col, end_dt)
                # print(f"     📦 Week {week_idx}: cutoff < {end_dt.strftime('%Y-%m-%d %H:%M:%S')}  "
                    #   f"(target window [{start_dt.strftime('%Y-%m-%d')} → {end_dt.strftime('%Y-%m-%d')}))")
                week_rows = list(fetch_all(entity, page_size=PAGE_SIZE, filter_exprs=before_variants))
                # print(f"        • Retrieved {len(week_rows)} rows")
                oeai.log.debug(
                    f"Week {week_idx}: cutoff < {end_dt.strftime('%Y-%m-%d %H:%M:%S')}"
                    f" (target window [{start_dt.strftime('%Y-%m-%d')} → {end_dt.strftime('%Y-%m-%d')}])"
                    f" Retrieved {len(week_rows)} rows",
                    entity=entity,
                    mapped_name=mapped_name,
                    mode=mode,
                    date_col=date_col,
                    wk_idx=wk_idx,
                    start_dt=start_dt,
                    end_dt=end_dt,
                    wk_rows=week_rows,
                )
                if week_rows:
                    all_rows.extend(week_rows)

            if not all_rows:
                # print(f"⚠️  No data returned for {entity} in the bounded period")
                oeai.log.debug(
                    f"⚠️  No data returned for {entity} in the bounded period",
                    entity=entity,
                    mapped_name=mapped_name,
                    mode=mode,
                )
                results.append({"table": entity, "saved_as": mapped_name, "rows_written": 0, "path": out_path, "status": "empty"})
                continue

            json_lines = [json.dumps(rec, default=str) for rec in all_rows]
            df = _json_lines_to_df(json_lines)

            if "_corrupt_record" in df.columns:
                df = df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

            # Guard-rail to the global bound + dedupe
            df = df.filter(
                (F.col(date_col) >= F.lit(BOUND_START_DATE.strftime("%Y-%m-%dT%H:%M:%S"))) &
                (F.col(date_col) <  F.lit(end_exclusive.strftime("%Y-%m-%dT%H:%M:%S")))
            ).dropDuplicates() \
             .withColumn("_ingested_at", F.current_timestamp())

            # print(f"💾 Writing (bounded weekly-before) {entity} → {mapped_name} (mode=overwrite)")
            oeai.log.debug(
                f"Writing (bounded weekly-before) {entity} → {mapped_name} (mode=overwrite)",
                entity=entity,
                mapped_name=mapped_name,
            )
            (
                df.write
                  .format("delta")
                  .mode("overwrite")
                  .option("overwriteSchema", "true")
                  .save(out_path)
            )
            rows_written = spark.read.format("delta").load(out_path).count()
            # print(f"✅ Completed {entity} → {mapped_name} (rows written: {rows_written:,})")
            oeai.log.info(
                f"Completed {entity} → {mapped_name} (rows written: {rows_written:,})",
                entity=entity,
                mapped_name=mapped_name,
                rows_written=rows_written,
            )
            results.append({
                "table": entity,
                "saved_as": mapped_name,
                "rows_written": rows_written,
                "path": out_path,
                "status": "ok" if rows_written > 0 else "empty"
            })

    else:
        # --- Full entities: one-shot fetch/write ---
        gen = fetch_all(entity, page_size=PAGE_SIZE, filter_exprs=None)
        rows = list(gen)
        if not rows:
            # print(f"⚠️  No data returned for {entity}")
            oeai.log.warning(
                f"⚠️  No data returned for {entity}",
                entity=entity,
            )
            results.append({"table": entity, "saved_as": mapped_name, "rows_written": 0, "path": out_path, "status": "empty"})
            continue

        json_lines = [json.dumps(rec, default=str) for rec in rows]
        df = _json_lines_to_df(json_lines)

        if "_corrupt_record" in df.columns:
            df = df.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")

        df = df.withColumn("_ingested_at", F.current_timestamp())

        # print(f"💾 Writing {entity} → {mapped_name} (rows={len(rows)}, mode=overwrite)")
        oeai.log.debug(
                f"Writing {entity} → {mapped_name} (mode=overwrite)",
                entity=entity,
                mapped_name=mapped_name,
            )
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .save(out_path)
        )
        rows_written = df.count()
        # print(f"✅ Completed {entity} → {mapped_name} (rows written: {rows_written:,})")
        oeai.log.info(
            f"Completed {entity} → {mapped_name} (rows written: {rows_written:,})",
            entity=entity,
            mapped_name=mapped_name,
            rows_written=rows_written,
        )
        results.append({
            "table": entity,
            "saved_as": mapped_name,
            "rows_written": rows_written,
            "path": out_path,
            "status": "ok" if rows_written > 0 else "empty"
        })

# print("\n🏁 All entities processed. Summary below:")
oeai.log.info("All entities processed.")
summary_df = spark.createDataFrame(results)
display(summary_df)

oeai.log.end_block()
oeai.log.checkpoint("Processing")

In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint("shutdown")